# Diagnóstico del entorno

**Corre esta celda antes que nada.** Comprueba en orden todo lo que puede fallar y
te da un informe completo.

No se detiene en el primer error: cada comprobación va por separado, así que un fallo
temprano no oculta los demás.

**Cuando termine, copia toda la salida y pégamela.** Con eso sé exactamente qué arreglar.

---

Si la celda ni siquiera arranca, el problema es el kernel:

1. Arriba a la derecha del notebook, botón **Seleccionar kernel**
2. **Entornos de Python** → **ml_venv**
3. Si `ml_venv` no aparece, es que VS Code no lo encuentra. Dímelo.

In [ ]:
# ===========================================================================
# DIAGNOSTICO DEL ENTORNO
#
# Escribe el informe COMPLETO en salidas/logs/diagnostico.txt y muestra aqui
# solo un resumen corto.
#
# POR QUE UN ARCHIVO: VS Code trunca las salidas largas de celda ("Output is
# truncated"), y justo lo que se pierde es el final, que es la parte que
# importa. Un archivo no se trunca.
# ===========================================================================

import sys, io, platform
from pathlib import Path

_buf = io.StringIO()          # aqui se acumula el informe completo
_resumen = []                 # lineas cortas que si se imprimen en pantalla

def log(*a):
    """Escribe solo en el informe."""
    print(*a, file=_buf)

def clave(etiqueta, valor, ok=None):
    """Escribe en el informe Y en el resumen de pantalla."""
    marca = "" if ok is None else ("  [OK]" if ok else "  [FALLA]")
    linea = f"{etiqueta:<28}: {valor}{marca}"
    log(linea); _resumen.append(linea)

log("=" * 70); log("INFORME DE DIAGNOSTICO"); log("=" * 70)

# --- 1. Interprete --------------------------------------------------------
log("\n[1] INTERPRETE")
clave("python", sys.version.split()[0])
clave("sistema", platform.platform())
log(f"ejecutable: {sys.executable}")
clave("kernel ml_venv", "ml_venv" in sys.executable, "ml_venv" in sys.executable)

# --- 2. Directorio y raiz -------------------------------------------------
log("\n[2] RUTAS")
cwd = Path.cwd()
log(f"directorio actual: {cwd}")

RAIZ = None
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "config.py").exists():
        RAIZ = cand
        break
clave("raiz del proyecto", RAIZ if RAIZ else "NO ENCONTRADA", RAIZ is not None)

if RAIZ is None:
    log("\nContenido del directorio actual:")
    for q in sorted(cwd.iterdir())[:30]:
        log(f"   {'[dir] ' if q.is_dir() else '      '}{q.name}")
else:
    sys.path.insert(0, str(RAIZ / "src"))

# --- 3. Archivos ----------------------------------------------------------
log("\n[3] ARCHIVOS DEL PROYECTO")
faltan = []
if RAIZ:
    for rel in ["src/config.py", "src/datos.py", "src/util.py", "src/graficos.py",
                "src/mapas.py", "notebooks/01_eda.ipynb", "requirements.txt"]:
        existe = (RAIZ / rel).exists()
        log(f"  {'OK   ' if existe else 'FALTA'}  {rel}")
        if not existe:
            faltan.append(rel)
clave("archivos que faltan", faltan if faltan else "ninguno", not faltan)

# --- 4. Bibliotecas -------------------------------------------------------
log("\n[4] BIBLIOTECAS")
sin_instalar = []
for nombre in ("pandas", "numpy", "pyarrow", "scipy", "sklearn",
               "statsmodels", "matplotlib", "plotly"):
    try:
        mod = __import__(nombre)
        log(f"  OK     {nombre:<14} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        log(f"  FALTA  {nombre:<14} {type(e).__name__}")
        sin_instalar.append(nombre)
clave("bibliotecas ausentes", sin_instalar if sin_instalar else "ninguna",
      not sin_instalar)
if sin_instalar:
    log(f"\n  Instalalas con:  pip install {' '.join(sin_instalar)}")

# --- 5. Modulos del proyecto ----------------------------------------------
log("\n[5] MODULOS DEL PROYECTO")
modulos_mal = []
if RAIZ:
    for nombre in ("config", "datos", "util", "graficos", "mapas"):
        try:
            __import__(nombre)
            log(f"  OK     {nombre}")
        except Exception as e:
            log(f"  ERROR  {nombre}: {type(e).__name__}: {e}")
            modulos_mal.append(nombre)
clave("modulos con error", modulos_mal if modulos_mal else "ninguno",
      not modulos_mal)

# --- 6. Ruta de la base ---------------------------------------------------
log("\n[6] BASE DE DATOS")
ruta_ok = False
try:
    from config import RUTA_DATOS
    ruta = Path(RUTA_DATOS)
    log(f"RUTA_DATOS configurada: {ruta}")
    ruta_ok = ruta.exists()
    clave("carpeta de datos existe", ruta_ok, ruta_ok)
    if ruta_ok:
        log("contenido:")
        for q in sorted(ruta.iterdir())[:25]:
            log(f"   {'[dir] ' if q.is_dir() else '      '}{q.name}")
        cm = (ruta / "cargar_matriz.py").exists()
        clave("cargar_matriz.py presente", cm, cm)
except Exception as e:
    clave("config.RUTA_DATOS", f"{type(e).__name__}: {e}", False)

# --- 7. Carga de la matriz ------------------------------------------------
log("\n[7] CARGA DE LA MATRIZ")
if ruta_ok:
    try:
        from datos import cargar_base
        print("cargando la matriz... esto puede tardar un par de minutos")
        df = cargar_base()
        clave("filas", f"{len(df):,}")
        clave("columnas", df.shape[1])
        clave("anios", f"{int(df['ANIO'].min())}-{int(df['ANIO'].max())}")
        log(f"\ncolumnas: {list(df.columns)}")
        log("\ntipos de las columnas clave:")
        for c in ["BPN", "ANIO", "COD_DPTO", "COD_MUNIC", "NIV_EDUM"]:
            if c in df.columns:
                ej = df[c].dropna()
                ej = repr(ej.iloc[0]) if len(ej) else "(toda nula)"
                log(f"   {c:<12} {str(df[c].dtype):<12} ejemplo: {ej}")
            else:
                log(f"   {c:<12} NO EXISTE")
                _resumen.append(f"  columna ausente: {c}")
    except Exception as e:
        import traceback
        clave("carga de la matriz", f"FALLO {type(e).__name__}: {e}", False)
        log("\n" + traceback.format_exc())
else:
    clave("carga de la matriz", "omitida, la ruta no existe", False)

# --- Escritura del informe -------------------------------------------------
destino = (RAIZ / "salidas" / "logs") if RAIZ else Path.cwd()
destino.mkdir(parents=True, exist_ok=True)
archivo = destino / "diagnostico.txt"
archivo.write_text(_buf.getvalue(), encoding="utf-8")

print("\n" + "=" * 62)
print("RESUMEN")
print("=" * 62)
for linea in _resumen:
    print(linea)
print("=" * 62)
print(f"Informe completo escrito en:\n  {archivo}")
print("\nAbre ese archivo, copia todo y pegamelo en el chat.")

**Qué hacer con esto.**

Copia toda la salida y pégamela en el chat. Con eso identifico el problema exacto en vez
de adivinar.

Los cuatro fallos más probables, por orden:

| Síntoma en el informe | Causa | Arreglo |
|---|---|---|
| `kernel ml_venv? : False` | Kernel equivocado | Seleccionar kernel → ml_venv |
| `RAIZ` no encontrada | Carpeta abierta equivocada, o zip descomprimido en un nivel de más | Abrir `CODIGO_VSC` en VS Code |
| `FALTA plotly` | Biblioteca ausente | `pip install plotly` en la terminal |
| Punto 7 dice `existe?: False` | `RUTA_DATOS` apunta mal | Corregir la ruta en `src/config.py` |